In [ ]:
# warnings keeps notebook output focused on results instead of library notices.
import warnings
# pandas works with table-shaped data such as CSV files.
import pandas as pd
# numpy supports numerical calculations and arrays.
import numpy as np
# matplotlib provides the base plotting tools.
import matplotlib.pyplot as plt
# seaborn makes statistical graphs easier to create and read.
import seaborn as sns
# Path helps create the report folder safely when it is missing.
from pathlib import Path

# train_test_split makes separate training and testing datasets.
from sklearn.model_selection import train_test_split, GridSearchCV
# StandardScaler puts continuous measurements on a comparable scale.
from sklearn.preprocessing import StandardScaler, OneHotEncoder
# ColumnTransformer applies different preparation steps to different columns.
from sklearn.compose import ColumnTransformer
# DummyClassifier supplies a simple no-skill baseline for comparison.
from sklearn.dummy import DummyClassifier
# LogisticRegression is a linear classification model.
from sklearn.linear_model import LogisticRegression
# KNeighborsClassifier predicts from the nearest similar customers.
from sklearn.neighbors import KNeighborsClassifier
# DecisionTreeClassifier learns rule-based splits in the data.
from sklearn.tree import DecisionTreeClassifier
# RandomForestClassifier combines many decision trees.
from sklearn.ensemble import RandomForestClassifier
# XGBClassifier is the gradient-boosted tree model used in this project.
from xgboost import XGBClassifier
# These functions calculate model-quality scores and evaluation plots.
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             confusion_matrix, RocCurveDisplay)

# Hide non-actionable third-party notices so a normal run is free of warning output.
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")
# Use one consistent visual style for every chart in the notebook.
sns.set_theme(style="whitegrid")
# Make the output folder if it does not already exist.
Path("../Report").mkdir(parents=True, exist_ok=True)


## 1. Data Loading and Understanding

This section reads the original customer file and checks its basic health before any modelling. The checks show the table size, column types, missing values, repeated rows, and summary values. Identifier fields are removed only after inspection because they identify people rather than describe churn behaviour.


In [ ]:
# Read the original customer churn data from the required project-relative path.
df = pd.read_csv("../Dataset/Churn_Modelling.csv")

# Display the number of customer records and columns in the original file.
print("Dataset dimensions (rows, columns):", df.shape)
# Display each column's data type so text and numeric fields are clear.
print("\nData types:")
print(df.dtypes)
# Count missing values in every column to identify incomplete data.
print("\nMissing values per column:")
print(df.isnull().sum())
# Count fully repeated customer records to check for duplicate rows.
print("\nDuplicate rows:", df.duplicated().sum())
# Summarise the numerical columns with count, mean, spread, and range values.
print("\nNumerical descriptive statistics:")
print(df.describe().T)

# Remove RowNumber because it is only a row position, not customer behaviour.
# Remove CustomerId because it is a unique identifier with no business meaning for churn.
# Remove Surname because names identify people and do not generalise as churn predictors.
df_clean = df.drop(columns=["RowNumber", "CustomerId", "Surname"])
# Confirm the remaining columns that will be used for EDA and modelling.
print("\nColumns retained for analysis:", df_clean.columns.tolist())


## 2. Exploratory Data Analysis (EDA)

EDA is completed on the readable, unscaled data so the graphs retain their original business meaning. The target plot checks class balance; the categorical plots compare churn across groups; the numerical plots show how customer values differ by churn status; and the correlation heatmap checks linear relationships among numeric variables.


In [ ]:
# Count retained and churned customers for the target-balance graph.
target_counts = df_clean["Exited"].value_counts().sort_index()
# Create a high-resolution chart that shows the target class balance.
plt.figure(figsize=(8, 6))
# Draw the bars using clear colours for retained and churned customers.
ax = sns.barplot(x=["Retained (0)", "Churned (1)"], y=target_counts.values,
                 palette=["#4C72B0", "#DD8452"])
# Write the exact count above each bar for easy interpretation.
for bar, count in zip(ax.patches, target_counts.values):
    ax.annotate(f"{count:,} ({count / len(df_clean):.1%})",
                (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                ha="center", va="bottom")
# Label the chart so its purpose is clear in the report.
plt.title("Customer Churn Target Distribution")
plt.xlabel("Customer outcome")
plt.ylabel("Number of customers")
# Save the first required EDA figure at report-quality resolution.
plt.savefig("../Report/EDA_Target_Distribution.png", dpi=300, bbox_inches="tight")
# Show the figure in the notebook before closing it.
plt.show()

# List the readable categories whose relationship with churn will be compared.
categorical_features = ["Geography", "Gender", "NumOfProducts", "IsActiveMember"]
# Create a two-by-two layout so all four category comparisons appear together.
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# Draw one churn-by-category count chart on each subplot.
for feature, ax in zip(categorical_features, axes.flat):
    # Separate the bars by target class to make churn differences visible.
    sns.countplot(data=df_clean, x=feature, hue="Exited", ax=ax,
                  palette={0: "#4C72B0", 1: "#DD8452"})
    # Give each small chart a descriptive title and axis labels.
    ax.set_title(f"Churn by {feature}")
    ax.set_xlabel(feature)
    ax.set_ylabel("Number of customers")
    # Rename the target legend into plain business language.
    ax.legend(title="Outcome", labels=["Retained (0)", "Churned (1)"])
# Adjust spacing so labels do not overlap.
plt.tight_layout()
# Save the second required EDA figure in the report folder.
plt.savefig("../Report/EDA_Categorical_Features.png", dpi=300, bbox_inches="tight")
# Show the completed category comparison figure.
plt.show()

# List the original continuous variables that will be compared by churn status.
numerical_eda_features = ["Age", "Balance", "CreditScore", "EstimatedSalary"]
# Create a two-by-two grid of boxplots for the four numerical variables.
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# Draw one distribution comparison for each numerical variable.
for feature, ax in zip(numerical_eda_features, axes.flat):
    # Use boxplots to reveal median, spread, and potential outliers by outcome.
    sns.boxplot(data=df_clean, x="Exited", y=feature, ax=ax,
                palette={0: "#4C72B0", 1: "#DD8452"})
    # Replace numeric target ticks with words that readers can understand.
    ax.set_xticks([0, 1], ["Retained (0)", "Churned (1)"])
    # Label each plot with the measure being inspected.
    ax.set_title(f"{feature} by Customer Outcome")
    ax.set_xlabel("Customer outcome")
    ax.set_ylabel(feature)
# Keep all boxplots readable when the figure is saved.
plt.tight_layout()
# Save the third required EDA figure in the report folder.
plt.savefig("../Report/EDA_Numerical_Distributions.png", dpi=300, bbox_inches="tight")
# Show the numerical comparison figure.
plt.show()

# Select only raw numeric columns because correlations require numeric values.
raw_numeric = df_clean.select_dtypes(include=np.number)
# Calculate pairwise correlations without encoding or scaling the source data.
correlation_matrix = raw_numeric.corr()
# Create a larger canvas so all heatmap labels remain legible.
plt.figure(figsize=(12, 9))
# Draw a labelled heatmap with a colour scale from negative to positive correlation.
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, square=True, linewidths=0.5)
# Explain exactly what the heatmap represents.
plt.title("Correlation Matrix of Raw Numerical Features")
# Save the fourth required EDA figure in the report folder.
plt.savefig("../Report/EDA_Correlation_Matrix.png", dpi=300, bbox_inches="tight")
# Show the correlation heatmap.
plt.show()


## 3. Data Preprocessing and Train-Test Split

The target is separated from the predictors and the data is split into 70% training and 30% testing data with the same churn proportion in each set. This prevents data leakage: the scaler and encoder are fitted only with `X_train`, so neither learns any information from the held-out test customers. The already-fitted steps are then used to transform `X_test`.


In [ ]:
# Separate all predictor columns from the churn target column.
X = df_clean.drop(columns="Exited")
# Store the binary churn outcome separately for classification.
y = df_clean["Exited"]

# Split customers into 70% training and 30% testing data while preserving churn balance.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# List continuous features that need scaling because their units and ranges differ.
continuous_features = ["CreditScore", "Age", "Tenure", "Balance", "NumOfProducts", "EstimatedSalary"]
# List text categories that must be converted to model-readable numbers.
categorical_features = ["Geography", "Gender"]

# Build a preparation step for each feature type.
preprocessor = ColumnTransformer(
    transformers=[
        # StandardScaler centres and scales only continuous training measurements.
        ("numeric", StandardScaler(), continuous_features),
        # OneHotEncoder converts categories to indicator columns for a model.
        # drop='first' removes one reference category to avoid the dummy-variable trap.
        ("categorical", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features),
    ],
    # Keep existing binary columns, such as HasCrCard and IsActiveMember, unchanged.
    remainder="passthrough",
)

# Fit the scaler and encoder only on training data, then prepare the training inputs.
X_train_prep = preprocessor.fit_transform(X_train)
# Apply the learned training-only transformations to unseen test customers.
X_test_prep = preprocessor.transform(X_test)
# Display the prepared shapes to confirm the same columns exist in both datasets.
print("Prepared training shape:", X_train_prep.shape)
print("Prepared testing shape:", X_test_prep.shape)


## 4. Model Development and Benchmarking

Six classifiers are evaluated on the same untouched test set. The dummy baseline shows the performance of always choosing the majority class, while the five machine-learning models show whether customer information adds useful predictive value. Accuracy measures all correct predictions; precision measures the reliability of churn alerts; recall measures how many actual churners are found; F1 balances precision and recall; ROC-AUC measures ranking quality across thresholds.


In [ ]:
# Store the baseline and five learning models in one named collection.
models = {
    # The dummy model always predicts the most common training outcome.
    "Baseline (Dummy)": DummyClassifier(strategy="most_frequent"),
    # The logistic model gives a linear probability-based benchmark.
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    # The KNN model compares each customer with nearby training customers.
    "K-Nearest Neighbors": KNeighborsClassifier(),
    # The tree model learns simple decision rules.
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    # The forest model averages many decision trees.
    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
    # The boosted-tree model learns sequentially from previous tree errors.
    "XGBoost": XGBClassifier(random_state=42, eval_metric="logloss", n_jobs=-1),
}
# Create an empty list that will hold one metric row per model.
benchmark_rows = []

# Train and evaluate every model using exactly the same prepared datasets.
for model_name, model in models.items():
    # Learn the relationship between prepared training features and churn labels.
    model.fit(X_train_prep, y_train)
    # Produce churn class predictions for the untouched test customers.
    y_pred = model.predict(X_test_prep)
    # Produce churn probabilities needed for the threshold-independent ROC-AUC score.
    y_probability = model.predict_proba(X_test_prep)[:, 1]
    # Save accuracy, precision, recall, F1, and ROC-AUC in a readable result row.
    benchmark_rows.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_probability),
    })

# Turn the collected metric rows into a side-by-side comparison table.
results_df = pd.DataFrame(benchmark_rows)
# Round displayed values without changing the underlying comparison calculation.
display(results_df.round(4))

# Reshape the metric columns so seaborn can draw a grouped comparison chart.
plot_results = results_df.melt(id_vars="Model", var_name="Metric", value_name="Score")
# Create a wide figure so all six model names remain readable.
plt.figure(figsize=(14, 7))
# Draw grouped bars to compare the five quality metrics across models.
sns.barplot(data=plot_results, x="Model", y="Score", hue="Metric", palette="Set2")
# Add a descriptive title and readable axis labels.
plt.title("Model Performance Comparison on the Test Set")
plt.xlabel("Model")
plt.ylabel("Score")
# Rotate labels slightly to prevent overlap.
plt.xticks(rotation=20, ha="right")
# Keep the score axis within its meaningful probability range.
plt.ylim(0, 1.05)
# Save the required overall comparison chart in the report folder.
plt.savefig("../Report/Model_Performance_Comparison.png", dpi=300, bbox_inches="tight")
# Show the comparison chart.
plt.show()


## 5. Hyperparameter Tuning and Imbalance Handling

Churn is the minority class, so a model can appear accurate while missing too many churners. The tuned Random Forest uses balanced class weights, and XGBoost gives extra weight to the minority churn class with `scale_pos_weight`. Grid search tests combinations of settings; five-fold cross-validation (`cv=5`) repeats this comparison over five different training/validation folds, giving a more dependable training-set choice before the final test evaluation.


In [ ]:
# Calculate the number of retained customers per churned customer in training data.
positive_weight = (y_train == 0).sum() / (y_train == 1).sum()

# Define the Random Forest settings that will be tested by grid search.
rf_param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [6, 12, 18],
    "min_samples_split": [2, 5],
}
# Create a five-fold search that favours good churn recall through F1 balance.
rf_grid = GridSearchCV(
    estimator=RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1),
    param_grid=rf_param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
)
# Test the Random Forest settings using only prepared training data.
rf_grid.fit(X_train_prep, y_train)
# Keep the best cross-validated Random Forest for final testing.
best_rf = rf_grid.best_estimator_

# Define the XGBoost settings that will be tested by grid search.
xgb_param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.03, 0.1],
}
# Create a five-fold search with minority-churn weighting for XGBoost.
xgb_grid = GridSearchCV(
    estimator=XGBClassifier(scale_pos_weight=positive_weight, random_state=42,
                            eval_metric="logloss", n_jobs=-1),
    param_grid=xgb_param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
)
# Test the XGBoost settings using only prepared training data.
xgb_grid.fit(X_train_prep, y_train)
# Keep the best cross-validated XGBoost model for final testing.
best_xgb = xgb_grid.best_estimator_

# Print the selected settings so the final models are reproducible.
print("Best Random Forest parameters:", rf_grid.best_params_)
print("Best XGBoost parameters:", xgb_grid.best_params_)

# Predict churn with the tuned Random Forest on the held-out test data.
y_pred_rf = best_rf.predict(X_test_prep)
# Predict churn with the tuned XGBoost model on the held-out test data.
y_pred_xgb = best_xgb.predict(X_test_prep)
# Print precision, recall, F1, and support for the tuned Random Forest.
print("\nTuned Random Forest classification report:")
print(classification_report(y_test, y_pred_rf, zero_division=0))
# Print precision, recall, F1, and support for the tuned XGBoost model.
print("Tuned XGBoost classification report:")
print(classification_report(y_test, y_pred_xgb, zero_division=0))


## 6. Results, Visualizations, and Feature Importance

The final figure presents both error patterns, each model's ability to separate churners across thresholds, and the features the tuned XGBoost model relied on most. For this churn task, true positives are particularly important because they identify customers the bank can try to retain.


In [ ]:
# Calculate test-set churn probabilities for the tuned Random Forest ROC curve.
y_probability_rf = best_rf.predict_proba(X_test_prep)[:, 1]
# Calculate test-set churn probabilities for the tuned XGBoost ROC curve.
y_probability_xgb = best_xgb.predict_proba(X_test_prep)[:, 1]
# Get the transformed feature names so importance scores can be labelled correctly.
feature_names = preprocessor.get_feature_names_out()
# Remove technical transformer prefixes to create report-friendly feature labels.
feature_names = [name.replace("numeric__", "").replace("categorical__", "")
                 .replace("remainder__", "") for name in feature_names]
# Pair each XGBoost importance score with its correct transformed feature name.
feature_importance = pd.Series(best_xgb.feature_importances_, index=feature_names)
# Keep the ten strongest features and order them for a horizontal bar chart.
top_features = feature_importance.sort_values(ascending=True).tail(10)

# Create a two-by-two master layout for the requested final interpretations.
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
# Draw the first confusion matrix for the tuned Random Forest.
sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt="d", cmap="Blues", ax=axes[0, 0])
# Label the Random Forest error matrix clearly.
axes[0, 0].set_title("Tuned Random Forest Confusion Matrix")
axes[0, 0].set_xlabel("Predicted outcome")
axes[0, 0].set_ylabel("Actual outcome")
# Draw the second confusion matrix for tuned XGBoost.
sns.heatmap(confusion_matrix(y_test, y_pred_xgb), annot=True, fmt="d", cmap="Greens", ax=axes[0, 1])
# Label the XGBoost error matrix clearly.
axes[0, 1].set_title("Tuned XGBoost Confusion Matrix")
axes[0, 1].set_xlabel("Predicted outcome")
axes[0, 1].set_ylabel("Actual outcome")
# Draw the Random Forest ROC curve and include its AUC in the legend.
RocCurveDisplay.from_predictions(y_test, y_probability_rf, name="Tuned Random Forest", ax=axes[1, 0])
# Draw the XGBoost ROC curve on the same axes for a direct comparison.
RocCurveDisplay.from_predictions(y_test, y_probability_xgb, name="Tuned XGBoost", ax=axes[1, 0])
# Add the no-skill diagonal reference line.
axes[1, 0].plot([0, 1], [0, 1], "k--", label="No-skill reference")
# Title and label the ROC comparison plot.
axes[1, 0].set_title("ROC Curves for Tuned Models")
axes[1, 0].set_xlabel("False Positive Rate")
axes[1, 0].set_ylabel("True Positive Rate")
axes[1, 0].legend(loc="lower right")
# Draw horizontal bars so the most influential XGBoost features are easy to compare.
axes[1, 1].barh(top_features.index, top_features.values, color="#55A868")
# Label the feature-importance chart for report readers.
axes[1, 1].set_title("Top 10 Tuned XGBoost Feature Importances")
axes[1, 1].set_xlabel("Relative importance")
axes[1, 1].set_ylabel("Prepared feature")
# Adjust subplot spacing before saving the master figure.
plt.tight_layout()
# Save all four required final visual interpretations in one report figure.
plt.savefig("../Report/Model_Evaluation_Plots.png", dpi=300, bbox_inches="tight")
# Show the master evaluation figure.
plt.show()


## 7. Conclusion and Findings

1. **Can customer demographics and transaction metrics reliably predict churn?** Yes. The benchmark results should exceed the majority-class dummy baseline, showing that demographic, account, product, and activity measures contain useful churn signals. However, the test-set scores also show that churn cannot be predicted perfectly, so the model should support—not replace—retention decisions.

2. **Which machine learning model performed best and why (Recall focus)?** Select the tuned model with the higher test-set churn recall from the classification reports above. In this imbalance-aware workflow, tuned XGBoost is expected to be the strongest recall-focused choice because `scale_pos_weight` makes missed churners more costly; its choice should be confirmed from the final printed recall rather than accuracy alone.

3. **Which features had the greatest predictive influence?** Use the final XGBoost feature-importance panel as the evidence. In this dataset, age, number of products, active-member status, balance, and geography commonly appear among the strongest predictors. The exact ranked features from the executed chart are the final project finding.
